# Project 1 – Incremental Spark ETL

Course: Big Data Management  
Student: BULAND KUMAR PRADHAN 

Goal:  
Build a Spark ETL pipeline that processes NYC Taxi trip records incrementally, applies cleaning rules, enriches the data with zone information, and writes a clean Parquet dataset.

In [1]:
import os
import json
import shutil

# Paths
OUTBOX_PATH = "data/outbox/trips_enriched.parquet"
MANIFEST_PATH = "state/manifest.json"

# 1) Remove previous output dataset if it exists
if os.path.exists(OUTBOX_PATH):
    shutil.rmtree(OUTBOX_PATH)
    print("Deleted old output:", OUTBOX_PATH)
else:
    print("No old output found.")

# 2) Reset manifest
os.makedirs("state", exist_ok=True)
with open(MANIFEST_PATH, "w", encoding="utf-8") as f:
    json.dump({"processed_files": {}}, f, indent=2)

print("Manifest reset:", MANIFEST_PATH)

# 3) Show current inbox files
print("\nInbox contents:")
for root, dirs, files in os.walk("data/inbox"):
    for name in sorted(files):
        print("-", os.path.join(root, name))

No old output found.
Manifest reset: state/manifest.json

Inbox contents:
- data/inbox/yellow_tripdata_2025-01.parquet
- data/inbox/yellow_tripdata_2025-02.parquet


## 1. Spark Session Setup

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("IncrementalTaxiETL") \
    .getOrCreate()

spark

In [3]:
spark.version

'4.1.0'

In [4]:
spark.range(10).count()

10

In [5]:
import os
import json

INBOX_DIR = "data/inbox"
OUTBOX_PATH = "data/outbox/trips_enriched.parquet"
MANIFEST_PATH = "state/manifest.json"

print("Inbox directory:", INBOX_DIR)
print("Output path:", OUTBOX_PATH)
print("Manifest path:", MANIFEST_PATH)

Inbox directory: data/inbox
Output path: data/outbox/trips_enriched.parquet
Manifest path: state/manifest.json


## 2. Manifest State Management

The manifest tracks which input files have already been processed.
This ensures the pipeline processes only new files on each run.

In [6]:
def load_manifest(path):
    
    if not os.path.exists(path):
        return {"processed_files": {}}
    
    with open(path, "r") as f:
        return json.load(f)

manifest = load_manifest(MANIFEST_PATH)

print("Manifest contents:")
print(manifest)

Manifest contents:
{'processed_files': {}}


In [7]:
def list_parquet_files(folder):

    files = [
        f for f in os.listdir(folder)
        if f.endswith(".parquet")
    ]
    
    return sorted(files)

inbox_files = list_parquet_files(INBOX_DIR)

print("Inbox files:")
for f in inbox_files:
    print("-", f)

Inbox files:
- yellow_tripdata_2025-01.parquet
- yellow_tripdata_2025-02.parquet


## 3. Detect New Files

The pipeline reads files from `data/inbox` and compares them with
the manifest to determine which files need to be processed.

In [8]:
processed_files = manifest.get("processed_files", {})

new_files = [
    f for f in inbox_files
    if f not in processed_files
]

print("New files detected:")
for f in new_files:
    print("-", f)

New files detected:
- yellow_tripdata_2025-01.parquet
- yellow_tripdata_2025-02.parquet


In [9]:
print("Total inbox files:", len(inbox_files))
print("New files to process:", len(new_files))

Total inbox files: 2
New files to process: 2


In [10]:
new_paths = [f"{INBOX_DIR}/{f}" for f in new_files]

print("Paths to read:")
for p in new_paths:
    print("-", p)

Paths to read:
- data/inbox/yellow_tripdata_2025-01.parquet
- data/inbox/yellow_tripdata_2025-02.parquet


In [11]:
df_raw = spark.read.parquet(*new_paths)

print("Raw row count:", df_raw.count())

Raw row count: 7052769


In [12]:
from pyspark.sql.functions import input_file_name, regexp_extract, lit
from datetime import datetime, timezone

df_raw = df_raw \
    .withColumn("source_file_path", input_file_name()) \
    .withColumn(
        "source_file",
        regexp_extract(input_file_name(), r'([^/]+$)', 1)
    ) \
    .withColumn(
        "ingested_at",
        lit(datetime.now(timezone.utc).isoformat())
    )

In [13]:
df_raw.select(
    "source_file",
    "source_file_path",
    "ingested_at"
).show(5, truncate=False)

+-------------------------------+----------------------------------------------------------------------------+--------------------------------+
|source_file                    |source_file_path                                                            |ingested_at                     |
+-------------------------------+----------------------------------------------------------------------------+--------------------------------+
|yellow_tripdata_2025-01.parquet|file:///home/jovyan/work/project1/data/inbox/yellow_tripdata_2025-01.parquet|2026-03-08T21:08:57.827541+00:00|
|yellow_tripdata_2025-01.parquet|file:///home/jovyan/work/project1/data/inbox/yellow_tripdata_2025-01.parquet|2026-03-08T21:08:57.827541+00:00|
|yellow_tripdata_2025-01.parquet|file:///home/jovyan/work/project1/data/inbox/yellow_tripdata_2025-01.parquet|2026-03-08T21:08:57.827541+00:00|
|yellow_tripdata_2025-01.parquet|file:///home/jovyan/work/project1/data/inbox/yellow_tripdata_2025-01.parquet|2026-03-08T21:08:57.827541

## 4. Read New Trip Data

In [14]:
from pyspark.sql.functions import col

df_typed = df_raw \
    .withColumn("pickup_ts", col("tpep_pickup_datetime")) \
    .withColumn("dropoff_ts", col("tpep_dropoff_datetime"))

In [15]:
df_typed.select(
    "pickup_ts",
    "dropoff_ts"
).show(5)

+-------------------+-------------------+
|          pickup_ts|         dropoff_ts|
+-------------------+-------------------+
|2025-01-01 00:18:38|2025-01-01 00:26:59|
|2025-01-01 00:32:40|2025-01-01 00:35:13|
|2025-01-01 00:44:04|2025-01-01 00:46:01|
|2025-01-01 00:14:27|2025-01-01 00:20:01|
|2025-01-01 00:21:34|2025-01-01 00:25:06|
+-------------------+-------------------+
only showing top 5 rows


In [16]:
from pyspark.sql.functions import to_date

df_typed = df_typed.withColumn(
    "pickup_date",
    to_date("pickup_ts")
)

In [17]:
df_typed.select("pickup_ts", "pickup_date").show(5)

+-------------------+-----------+
|          pickup_ts|pickup_date|
+-------------------+-----------+
|2025-01-01 00:18:38| 2025-01-01|
|2025-01-01 00:32:40| 2025-01-01|
|2025-01-01 00:44:04| 2025-01-01|
|2025-01-01 00:14:27| 2025-01-01|
|2025-01-01 00:21:34| 2025-01-01|
+-------------------+-----------+
only showing top 5 rows


In [18]:
from pyspark.sql.functions import month

df_typed = df_typed.withColumn(
    "pickup_month",
    month("pickup_ts")
)

In [19]:
df_typed.select("pickup_ts", "pickup_month").show(5)

+-------------------+------------+
|          pickup_ts|pickup_month|
+-------------------+------------+
|2025-01-01 00:18:38|           1|
|2025-01-01 00:32:40|           1|
|2025-01-01 00:44:04|           1|
|2025-01-01 00:14:27|           1|
|2025-01-01 00:21:34|           1|
+-------------------+------------+
only showing top 5 rows


In [20]:
from pyspark.sql.functions import unix_timestamp, round

df_typed = df_typed.withColumn(
    "trip_duration_minutes",
    round(
        (unix_timestamp("dropoff_ts") - unix_timestamp("pickup_ts")) / 60,
        2
    )
)

In [21]:
df_typed.select(
    "pickup_ts",
    "dropoff_ts",
    "trip_duration_minutes"
).show(5)

+-------------------+-------------------+---------------------+
|          pickup_ts|         dropoff_ts|trip_duration_minutes|
+-------------------+-------------------+---------------------+
|2025-01-01 00:18:38|2025-01-01 00:26:59|                 8.35|
|2025-01-01 00:32:40|2025-01-01 00:35:13|                 2.55|
|2025-01-01 00:44:04|2025-01-01 00:46:01|                 1.95|
|2025-01-01 00:14:27|2025-01-01 00:20:01|                 5.57|
|2025-01-01 00:21:34|2025-01-01 00:25:06|                 3.53|
+-------------------+-------------------+---------------------+
only showing top 5 rows


In [22]:
df_typed.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)
 |-- source_file_path: string (nullable = false)
 |-- source_file: string (n

## 5. Scenario: Passenger Count Imputation

If `passenger_count` is null or zero, it is replaced with the median
passenger count for the same calendar month within the same source file.

In [23]:
from pyspark.sql.functions import col

invalid_passenger_rows = df_typed.filter(
    (col("passenger_count").isNull()) | (col("passenger_count") == 0)
)

print("Invalid passenger_count rows:", invalid_passenger_rows.count())

Invalid passenger_count rows: 1393493


In [24]:
from pyspark.sql.functions import expr

median_passenger_by_month = df_typed \
    .filter(col("passenger_count") > 0) \
    .groupBy("pickup_month") \
    .agg(
        expr("percentile_approx(passenger_count, 0.5)").alias("median_passenger_count")
    )

median_passenger_by_month.show()

+------------+----------------------+
|pickup_month|median_passenger_count|
+------------+----------------------+
|          12|                     1|
|           1|                     1|
|           2|                     1|
|           3|                     1|
+------------+----------------------+



In [25]:
df_with_median = df_typed.join(
    median_passenger_by_month,
    on="pickup_month",
    how="left"
)

In [26]:
df_with_median.select(
    "pickup_month",
    "passenger_count",
    "median_passenger_count"
).show(5)

+------------+---------------+----------------------+
|pickup_month|passenger_count|median_passenger_count|
+------------+---------------+----------------------+
|           1|              1|                     1|
|           1|              1|                     1|
|           1|              1|                     1|
|           1|              3|                     1|
|           1|              3|                     1|
+------------+---------------+----------------------+
only showing top 5 rows


In [27]:
from pyspark.sql.functions import when

df_imputed = df_with_median.withColumn(
    "passenger_count",
    when(
        (col("passenger_count").isNull()) | (col("passenger_count") == 0),
        col("median_passenger_count")
    ).otherwise(col("passenger_count"))
)

In [28]:
remaining_invalid = df_imputed.filter(
    (col("passenger_count").isNull()) | (col("passenger_count") == 0)
)

print("Remaining invalid passenger_count rows:", remaining_invalid.count())

Remaining invalid passenger_count rows: 0


In [29]:
df_imputed = df_imputed.drop("median_passenger_count")

## 6. Data Cleaning

Invalid records such as negative trip durations are removed.

In [30]:
input_row_count = df_imputed.count()

print("Row count before cleaning:", input_row_count)

Row count before cleaning: 7052769


In [31]:
negative_duration_rows = df_imputed.filter(
    col("trip_duration_minutes") < 0
)

print("Rows with negative trip duration:", negative_duration_rows.count())

Rows with negative trip duration: 217


In [32]:
df_clean = df_imputed.filter(
    col("trip_duration_minutes") >= 0
)

In [33]:
clean_row_count = df_clean.count()

print("Row count after cleaning:", clean_row_count)

Row count after cleaning: 7052552


In [34]:
removed_rows = input_row_count - clean_row_count

print("Rows removed during cleaning:", removed_rows)

Rows removed during cleaning: 217


## 7. Deduplication

Duplicate trips are removed using a key consisting of:

- VendorID
- pickup_ts
- dropoff_ts
- PULocationID
- DOLocationID
- trip_distance

In [35]:
before_dedup_count = df_clean.count()

print("Row count before deduplication:", before_dedup_count)

Row count before deduplication: 7052552


In [36]:
dedup_keys = [
    "pickup_ts",
    "dropoff_ts",
    "PULocationID",
    "DOLocationID",
    "passenger_count",
    "trip_distance"
]

df_dedup = df_clean.dropDuplicates(dedup_keys)

In [37]:
after_dedup_count = df_dedup.count()

print("Row count after deduplication:", after_dedup_count)

Row count after deduplication: 6951037


In [38]:
duplicates_removed = before_dedup_count - after_dedup_count

print("Duplicate rows removed:", duplicates_removed)

Duplicate rows removed: 101515


## 8. Data Enrichment

Pickup and dropoff zones are added using the taxi zone lookup table.

In [39]:
zone_lookup = spark.read.parquet("data/taxi_zone_lookup.parquet")

zone_lookup.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows


In [40]:
from pyspark.sql.functions import col

pickup_zones = zone_lookup.select(
    col("LocationID").alias("PULocationID"),
    col("Zone").alias("pickup_zone")
)

In [41]:
dropoff_zones = zone_lookup.select(
    col("LocationID").alias("DOLocationID"),
    col("Zone").alias("dropoff_zone")
)

In [42]:
df_enriched = df_dedup.join(
    pickup_zones,
    on="PULocationID",
    how="left"
)

In [43]:
df_enriched = df_enriched.join(
    dropoff_zones,
    on="DOLocationID",
    how="left"
)

In [44]:
df_enriched.select(
    "pickup_zone",
    "dropoff_zone"
).show(5)

+--------------------+--------------------+
|         pickup_zone|        dropoff_zone|
+--------------------+--------------------+
|Sutton Place/Turt...|            Gramercy|
|     Lenox Hill West|            Gramercy|
|Upper East Side S...|         Murray Hill|
|        Midtown East|Sutton Place/Turt...|
|Sutton Place/Turt...|            Gramercy|
+--------------------+--------------------+
only showing top 5 rows


## 9. Write Final Dataset

In [45]:
import shutil

shutil.rmtree("data/outbox/trips_enriched.parquet", ignore_errors=True)

print("Old output deleted")

Old output deleted


In [46]:
OUTPUT_PATH = "data/outbox/trips_enriched.parquet"

df_enriched \
    .coalesce(4) \
    .write \
    .mode("append") \
    .parquet(OUTPUT_PATH)

In [47]:
output_df = spark.read.parquet(OUTPUT_PATH)

print("Output rows:", output_df.count())

Output rows: 6951037


## 10. Update Manifest

In [48]:
manifest["processed_files"] = {
    f: {
        "processed_at": str(datetime.now()),
        "rows_written": clean_row_count
    }
    for f in new_files
}

with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2)

print("Manifest updated with processed files")

Manifest updated with processed files


## Section 11 — Performance Measurement

In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("TaxiETL") \
    .getOrCreate()

print("Spark restarted")

Spark restarted


In [2]:
import time

start_time = time.time()

df = spark.read.parquet("data/outbox/trips_enriched.parquet")
row_count = df.count()

end_time = time.time()

runtime = end_time - start_time

print("Rows:", row_count)
print("Runtime:", round(runtime,2), "seconds")

Rows: 6951037
Runtime: 3.7 seconds


In [3]:
import time

start = time.time()

df = spark.read.parquet("data/outbox/trips_enriched.parquet").cache()
rows = df.count()

end = time.time()

cache_time = end - start

print("Rows:", rows)
print("Runtime with caching:", round(cache_time,2), "seconds")

Rows: 6951037
Runtime with caching: 24.09 seconds


In [4]:
spark.conf.set("spark.sql.shuffle.partitions","8")

import time

start = time.time()

df = spark.read.parquet("data/outbox/trips_enriched.parquet")
rows = df.count()

end = time.time()

shuffle_time = end - start

print("Rows:", rows)
print("Runtime with reduced shuffle partitions:", round(shuffle_time,2), "seconds")

Rows: 6951037
Runtime with reduced shuffle partitions: 1.03 seconds
